# Digital Forge Video Renderer
This notebook automates the setup and rendering pipeline for the **digital_forge** HTML-to-video rendering engine inside a Google Colab GPU environment (T4 recommended).

Instead of manual uploads, this version clones your repository directly via Git.

In [ ]:
#@title Clone Repository & Directory Setup
import os

# 1. Safely jump back to the stable root directory to resolve the "ghost directory" getcwd error
%cd /content

# 2. Clean up existing directory now that we are outside of it
!rm -rf /content/digital-forge-reel

# 3. Clone the repository
repo_url = 'https://github.com/Vtheonly/digital_forge' #@param {type:'string'}
branch = 'main' #@param {type:'string'}
!git clone -b {branch} {repo_url} /content/digital-forge-reel

# 4. Change directory to the fresh repository root
%cd /content/digital-forge-reel

In [ ]:
#@title Install System & Node Dependencies
# Runs the modified setup-colab.sh script directly from the cloned repository
!bash setup-colab.sh

In [ ]:
#@title Download & Apply Fade Filters to YouTube Audio
import os

# Install yt-dlp to download the audio stream directly
!pip install -q yt-dlp

youtube_url = 'https://youtu.be/Op_fUZf_KQc' #@param {type:'string'}

# Ensure audio directory exists
os.makedirs('audio', exist_ok=True)

# 1. Download highest quality audio format directly as a WAV
!yt-dlp -x --audio-format wav -o "audio/downloaded.wav" "{youtube_url}"

# 2. Extract 31 seconds, apply 1.5s fade-in at start, and 2.0s fade-out at end (starting at t=29s)
!ffmpeg -y -i "audio/downloaded.wav" -ss 00:00:00 -t 31 -af "afade=t=in:ss=0:d=1.5,afade=t=out:st=29:d=2" "audio/forge_theme.wav"

# Clean up raw download file
if os.path.exists("audio/downloaded.wav"):
  os.remove("audio/downloaded.wav")

print("✓ Custom audio processed and saved to audio/forge_theme.wav")

In [ ]:
#@title GPU Diagnostics & Verification
# Verifies drivers, Vulkan configuration, and headless GPU rasterization
!node bin/forge-gpu-check --scene scenes/digital-forge-reel-en.html

In [ ]:
#@title Render English Video Reel (Smooth True 60fps)
# High-efficiency GPU-accelerated capture and NVENC encoding at native 60fps
!node bin/forge-render scenes/digital-forge-reel-en.html \
    --output output/en_cinematic.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder nvenc \
    --workers 2 \
    --fps 60 \
    --target-fps 60

In [ ]:
#@title Render Arabic (Darija) Video Reel (Smooth True 60fps)
!node bin/forge-render scenes/digital-forge-reel-ar.html \
    --output output/ar_cinematic.mp4 \
    --music audio/forge_theme.wav \
    --gpu \
    --encoder nvenc \
    --workers 2 \
    --fps 60 \
    --target-fps 60

In [ ]:
#@title Download Rendered Files
from google.colab import files
import os

print('Generated video files in output directory:')
if os.path.exists('output'):
  outputs = [f for f in os.listdir('output') if f.endswith('.mp4')]
  for out in outputs:
    print(f'- {out}')

# Uncomment to automatically trigger download of output files to your local machine
# if os.path.exists('output/en_cinematic.mp4'):
#   files.download('output/en_cinematic.mp4')
# if os.path.exists('output/ar_cinematic.mp4'):
#   files.download('output/ar_cinematic.mp4')